In [1]:
import keras.backend as K
import os
import numpy as np
import pylab as plt
from keras.utils import to_categorical
from keras.models import Model
from keras.layers import Input
from keras.layers import LSTM
from keras.layers import Dense
from keras.layers.convolutional import Conv3D
from keras.layers.convolutional_recurrent import ConvLSTM2D
from keras.layers.normalization import BatchNormalization
from keras.models import load_model
from keras.models import load_model
from keras.callbacks import EarlyStopping
from keras.callbacks import ModelCheckpoint
from keras.optimizers import Adam
import matplotlib.pyplot as plt
from keras.utils import to_categorical
from keras.regularizers import l2
from keras.backend import clip 
import math

In [2]:
import os
from keras.utils import multi_gpu_model
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID" 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

In [3]:
data = np.load('NPY_Files/data_with_frame_splits.npy')
data.shape

(657, 20, 128, 110, 1)

In [4]:
from sklearn.preprocessing import minmax_scale

shape = data.shape
data = minmax_scale(data.ravel(), feature_range=(0,255)).reshape(shape)
data.shape

(657, 20, 128, 110, 1)

In [5]:
# X1_precipitation = data[:100,:16,:,:,:]
# X1_precipitation.shape

# tem = np.zeros((100,1,128,110,1))
# #tem = tem.astype(int)

# X2_precipitation = np.concatenate((tem, data[:100,16:19,:,:,:]), axis=1)
# #X2_precipitation = X2_precipitation.astype('uint8')
# X2_precipitation.shape


# y_precipitation = data[:100,16:20,:,:,:]
# y_precipitation.shape
    
# del tem

In [6]:
"""
LSTM 2-layer 2048 nodes
"""

def define_models_lstm(n_input, n_output, n_unit):
    # define training encoder
    encoder_inputs = Input(shape=(None, n_input))
    encoder_1 = LSTM(n_unit, return_sequences=True, return_state= True)
    encoder_outputs_1, state_h_1, state_c_1 = encoder_1(encoder_inputs)
    encoder_states_1 = [state_h_1, state_c_1]
    encoder_2 = LSTM(n_unit, return_state=True)
    encoder_outputs_2, state_h_2, state_c_2 = encoder_2(encoder_outputs_1)
    encoder_states_2 = [state_h_2, state_c_2]
    # define training decoder
    decoder_inputs = Input(shape=(None, n_output))
    decoder_lstm_1 = LSTM(n_unit, return_sequences=True, return_state=True)
    decoder_outputs_1,_,_ = decoder_lstm_1(decoder_inputs, initial_state=encoder_states_1)
    decoder_lstm_2 = LSTM(n_unit, return_sequences=True, return_state=True)
    decoder_outputs_2,_,_ = decoder_lstm_2(decoder_outputs_1, initial_state=encoder_states_2)
    decoder_dense = Dense(n_output, activation='relu')
    decoder_outputs = decoder_dense(decoder_outputs_2)
    
    model = Model([encoder_inputs,decoder_inputs], decoder_outputs)  # training model
    print(model.summary(line_length=250))
    
    # define inference encoder
    encoder_model = Model(encoder_inputs, encoder_states_1 + encoder_states_2)
    #print(encoder_model.summary())
    
    # define inference decoder
    decoder_state_input_h_1 = Input(shape=(n_unit,))
    decoder_state_input_c_1 = Input(shape=(n_unit,))
    decoder_state_inputs_1 = [decoder_state_input_h_1, decoder_state_input_c_1]
    decoder_state_input_h_2 = Input(shape=(n_unit,))
    decoder_state_input_c_2 = Input(shape=(n_unit,))
    decoder_state_inputs_2 = [decoder_state_input_h_2, decoder_state_input_c_2]    
    decoder_outputs_1, state_h_1, state_c_1 = decoder_lstm_1(decoder_inputs, initial_state = decoder_state_inputs_1)
#    decoder_states_1 = [state_h_1, state_c_1]
    decoder_outputs_2, state_h_2, state_c_2 = decoder_lstm_2(decoder_outputs_1, initial_state=decoder_state_inputs_2)
#    decoder_states_2 = [state_h_2, state_c_2]
    decoder_outputs = decoder_dense(decoder_outputs_2)
    
    decoder_model = Model([decoder_inputs] + decoder_state_inputs_1 + decoder_state_inputs_2, decoder_outputs)
    print(decoder_model.summary())
    return model, encoder_model, decoder_model   

In [7]:
train_lstm, infenc_lstm, infdec_lstm = define_models_lstm(128*110, 128*110, 2048)  

X1_lstm = data[:100,:10,:,:,:].reshape(100,10,128*110)
X1_lstm.shape
tem = np.zeros((100,1,128*110))
x = tem
x.shape
del tem

X2_lstm = np.concatenate((x, data[:100,16:19,:,:,:].reshape(100,3,128*110)), axis=1) 
X2_lstm.shape

y_lstm = data[:100,16:20,:,:,:].reshape((100,4,128*110))
y_lstm.shape

# X1_lstm = data[:,:10,:,:,:].reshape(5485,10,101*101)
# X1_lstm.shape
# tem = np.zeros((5485,1,101*101))
# x = tem.astype(int)
# x.shape
# del tem
# X2_lstm = np.concatenate((x, data[:,10:14,:,:,:].reshape(5485,4,101*101)), axis=1) 
# X2_lstm.shape
# y_lstm = data[:,10:15,:,:,:].reshape((5485,5,101*101))
# y_lstm.shape



# X1_precipitation = data[:100,:10,:,:,:].reshape(100,10,128*110)
# X1_precipitation.shape

# tem = np.zeros((100,1,128,110,1))
# #tem = tem.astype(int)

# X2_precipitation = np.concatenate((tem, data[:100,16:19,:,:,:]), axis=1)
# #X2_precipitation = X2_precipitation.astype('uint8')
# X2_precipitation.shape


# y_precipitation = data[:100,16:20,:,:,:]
# y_precipitation.shape
    
# del tem

filepath = "saved-2048_Nodes-2Layer-{epoch:02d}.h5"
cp = ModelCheckpoint(filepath, verbose=1, save_best_only=False,mode='max', period=20)
train_lstm.compile(loss='mse', optimizer='adam', metrics=['mae'])
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)
history_lstm = train_lstm.fit([X1_lstm, X2_lstm],y_lstm, batch_size=32, validation_split=0.25, epochs=500, callbacks=[cp])


Model: "model"
__________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________
Layer (type)                                                                      Output Shape                                           Param #                       Connected to                                                                       
input_1 (InputLayer)                                                              [(None, None, 14080)]                                  0                                                                                                                
________________________________________________________________________________________________________________________________________________________________________________________________________________________________________

KeyboardInterrupt: 